In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import re
import os

titanic = sns.load_dataset('titanic')

# Task 1.3.1: Sort Titanic by fare descending. Who are the top 10 highest-paying passengers? Print their name (if available), fare, and class.
top_10 = titanic.sort_values(by='fare', ascending=False).head(10)
print("Top 10 highest paying:")
print(top_10[['who', 'fare', 'pclass']])

Top 10 highest paying:
       who      fare  pclass
679    man  512.3292       1
258  woman  512.3292       1
737    man  512.3292       1
88   woman  263.0000       1
438    man  263.0000       1
341  woman  263.0000       1
27     man  263.0000       1
742  woman  262.3750       1
311  woman  262.3750       1
299  woman  247.5208       1


In [2]:
# Task 1.3.2: Sort by pclass ascending, then by age descending within each class. Show the youngest in each class.
sorted_df = titanic.sort_values(by=['pclass', 'age'], ascending=[True, False])
print("Sorted sample:\n", sorted_df[['pclass', 'age']].head())
youngest = titanic.groupby('pclass')['age'].min()
print("Youngest in each class:\n", youngest)

Sorted sample:
      pclass   age
630       1  80.0
96        1  71.0
493       1  71.0
745       1  70.0
54        1  65.0
Youngest in each class:
 pclass
1    0.92
2    0.67
3    0.42
Name: age, dtype: float64


In [3]:
# Task 1.3.3: Add a rank column to a 10-student marks DataFrame (rank 1=best). Try all 5 rank methods and compare results for tied students.
marks_df = pd.DataFrame({'Student': list('ABCDEFGHIJ'), 'Marks': [80, 95, 80, 95, 80, 85, 90, 75, 40, 85]})
# ascending=False makes 1 the best rank
marks_df['Rank_min'] = marks_df['Marks'].rank(method='min', ascending=False)
marks_df['Rank_max'] = marks_df['Marks'].rank(method='max', ascending=False)
marks_df['Rank_dense'] = marks_df['Marks'].rank(method='dense', ascending=False)
marks_df['Rank_avg'] = marks_df['Marks'].rank(method='average', ascending=False)
marks_df['Rank_first'] = marks_df['Marks'].rank(method='first', ascending=False)

print(marks_df.sort_values(by='Marks', ascending=False))

  Student  Marks  Rank_min  Rank_max  Rank_dense  Rank_avg  Rank_first
1       B     95       1.0       2.0         1.0       1.5         1.0
3       D     95       1.0       2.0         1.0       1.5         2.0
6       G     90       3.0       3.0         2.0       3.0         3.0
9       J     85       4.0       5.0         3.0       4.5         5.0
5       F     85       4.0       5.0         3.0       4.5         4.0
0       A     80       6.0       8.0         4.0       7.0         6.0
4       E     80       6.0       8.0         4.0       7.0         8.0
2       C     80       6.0       8.0         4.0       7.0         7.0
7       H     75       9.0       9.0         5.0       9.0         9.0
8       I     40      10.0      10.0         6.0      10.0        10.0


In [4]:
# Task 2.3.1: On Titanic: compute survival rate by sex, by pclass, and by (sex x pclass) combination. Which group had the highest survival rate?
print("Survival by sex:\n", titanic.groupby('sex')['survived'].mean())
print("Survival by class:\n", titanic.groupby('pclass')['survived'].mean())

combo = titanic.groupby(['sex', 'pclass'])['survived'].mean()
print("Survival by sex and class:\n", combo.sort_values(ascending=False))

Survival by sex:
 sex
female    0.742038
male      0.188908
Name: survived, dtype: float64
Survival by class:
 pclass
1    0.629630
2    0.472826
3    0.242363
Name: survived, dtype: float64
Survival by sex and class:
 sex     pclass
female  1         0.968085
        2         0.921053
        3         0.500000
male    1         0.368852
        2         0.157407
        3         0.135447
Name: survived, dtype: float64


In [5]:
# Task 2.3.2: For each passenger class, find the mean, median, and std of fares. Use a single .agg() call.
fares_stats = titanic.groupby('pclass')['fare'].agg(['mean', 'median', 'std'])
print(fares_stats)

             mean   median        std
pclass                               
1       84.154687  60.2875  78.380373
2       20.662183  14.2500  13.417399
3       13.675550   8.0500  11.778142


In [6]:
# Task 2.3.3: Add a column class_median_age = median age within each pclass using transform. Then add a flag IsOlderThanClassMedian.
titanic['class_median_age'] = titanic.groupby('pclass')['age'].transform('median')
titanic['IsOlderThanClassMedian'] = titanic['age'] > titanic['class_median_age']
print(titanic[['pclass', 'age', 'class_median_age', 'IsOlderThanClassMedian']].head(8))

   pclass   age  class_median_age  IsOlderThanClassMedian
0       3  22.0              24.0                   False
1       1  38.0              37.0                    True
2       3  26.0              24.0                    True
3       1  35.0              37.0                   False
4       3  35.0              24.0                    True
5       3   NaN              24.0                   False
6       1  54.0              37.0                    True
7       3   2.0              24.0                   False


In [7]:
# Task 2.3.4: Filter out passenger classes with fewer than 200 passengers. How many passengers remain?
pclass_counts = titanic.groupby('pclass')['survived'].transform('count')
filtered_df = titanic[pclass_counts >= 200]
print("Passengers remaining:", len(filtered_df))

Passengers remaining: 707


In [8]:
# Task 2.3.5: Group by embarked port and compute: total passengers, survival rate, average fare. Sort by survival rate descending.
embarked_stats = titanic.groupby('embarked').agg(
    total_passengers=('survived', 'count'),
    survival_rate=('survived', 'mean'),
    average_fare=('fare', 'mean')
)
print(embarked_stats.sort_values(by='survival_rate', ascending=False))

          total_passengers  survival_rate  average_fare
embarked                                               
C                      168       0.553571     59.954144
Q                       77       0.389610     13.276030
S                      644       0.336957     27.079812


In [9]:
# Task 3.3.1: Create a Students table (id, name, age) and a Grades table (id, subject, marks). Perform all 4 join types and explain the row count difference.
students = pd.DataFrame({'id': [1, 2, 3], 'name': ['Alice', 'Bob', 'Charlie'], 'age': [20, 21, 22]})
grades = pd.DataFrame({'id': [2, 3, 4], 'subject': ['Math', 'Sci', 'Eng'], 'marks': [90, 85, 88]})

print("Inner:\n", pd.merge(students, grades, on='id', how='inner')) # only matching IDs (2, 3)
print("Left:\n", pd.merge(students, grades, on='id', how='left')) # all students, grade missing for 1
print("Right:\n", pd.merge(students, grades, on='id', how='right')) # all grades, student missing for 4
print("Outer:\n", pd.merge(students, grades, on='id', how='outer')) # all records from both

Inner:
    id     name  age subject  marks
0   2      Bob   21    Math     90
1   3  Charlie   22     Sci     85
Left:
    id     name  age subject  marks
0   1    Alice   20     NaN    NaN
1   2      Bob   21    Math   90.0
2   3  Charlie   22     Sci   85.0
Right:
    id     name   age subject  marks
0   2      Bob  21.0    Math     90
1   3  Charlie  22.0     Sci     85
2   4      NaN   NaN     Eng     88
Outer:
    id     name   age subject  marks
0   1    Alice  20.0     NaN    NaN
1   2      Bob  21.0    Math   90.0
2   3  Charlie  22.0     Sci   85.0
3   4      NaN   NaN     Eng   88.0


In [10]:
# Task 3.3.2: You have two monthly sales DataFrames (same columns). Concatenate them into a single yearly DataFrame. Add a Month column before concatenating.
sales_jan = pd.DataFrame({'Item': ['X', 'Y'], 'Qty': [100, 200]})
sales_feb = pd.DataFrame({'Item': ['X', 'Z'], 'Qty': [150, 50]})

sales_jan['Month'] = 'Jan'
sales_feb['Month'] = 'Feb'

yearly_sales = pd.concat([sales_jan, sales_feb], ignore_index=True)
print(yearly_sales)

  Item  Qty Month
0    X  100   Jan
1    Y  200   Jan
2    X  150   Feb
3    Z   50   Feb


In [11]:
# Task 3.3.3: Merge Titanic with a separate DataFrame mapping pclass -> class_label ('First','Second','Third'). Verify all rows have a label.
labels_df = pd.DataFrame({'pclass': [1, 2, 3], 'class_label': ['First', 'Second', 'Third']})
titanic_merged = pd.merge(titanic, labels_df, on='pclass', how='left')
print(titanic_merged[['pclass', 'class_label']].head())
print("Null labels:", titanic_merged['class_label'].isnull().sum())

   pclass class_label
0       3       Third
1       1       First
2       3       Third
3       1       First
4       3       Third
Null labels: 0


In [12]:
# Task 3.3.4: Detect duplicate rows after a merge. Use df.duplicated() to find and df.drop_duplicates() to remove them.
df_dup = pd.DataFrame({'id': [1, 2, 2, 3], 'val': ['a', 'b', 'b', 'c']})
print("Duplicates found:\n", df_dup[df_dup.duplicated()])
df_clean = df_dup.drop_duplicates()
print("After drop:\n", df_clean)

Duplicates found:
    id val
2   2   b
After drop:
    id val
0   1   a
1   2   b
3   3   c


In [13]:
# Task 4.3.1: Build a pivot table: mean and std of age grouped by pclass (rows) and survived (columns). Include totals.
pt_age = pd.pivot_table(titanic, values='age', index='pclass', columns='survived', aggfunc=['mean', 'std'], margins=True)
print(pt_age)

               mean                              std                      
survived          0          1        All          0          1        All
pclass                                                                    
1         43.695312  35.368197  38.233441  15.284243  13.760017  14.802856
2         33.544444  25.901566  29.877630  12.151581  14.837787  14.001077
3         26.555556  20.646118  25.140620  12.334882  11.995047  12.495398
All       30.626179  28.343690  29.699118  14.172110  14.950952  14.526497


In [14]:
# Task 4.3.2: Use crosstab to show percentage survival by sex AND embarked port. Which combination had the highest survival rate?
ct_surv = pd.crosstab(titanic['sex'], titanic['embarked'], values=titanic['survived'], aggfunc='mean') * 100
print("Survival %:\n", ct_surv)

Survival %:
 embarked          C          Q          S
sex                                      
female    87.671233  75.000000  68.965517
male      30.526316   7.317073  17.460317


In [15]:
# Task 4.3.3: Pivot Titanic to show total fare collected per class per embarkation port. Which port generated the most revenue in each class?
pt_fare = pd.pivot_table(titanic, values='fare', index='pclass', columns='embarked', aggfunc='sum')
print("Total fares:\n", pt_fare)
print("Most revenue port for each class:\n", pt_fare.idxmax(axis=1))

Total fares:
 embarked          C         Q          S
pclass                                  
1         8901.0750  180.0000  8936.3375
2          431.0917   37.0500  3333.7000
3          740.1295  805.2043  5169.3613
Most revenue port for each class:
 pclass
1    S
2    S
3    S
dtype: str


In [16]:
# Task 5.2.1: Use map to replace Titanic 'embarked' codes (C/Q/S) with full names (Cherbourg/Queenstown/Southampton).
port_map = {'C': 'Cherbourg', 'Q': 'Queenstown', 'S': 'Southampton'}
titanic['embarked_full'] = titanic['embarked'].map(port_map)
print(titanic[['embarked', 'embarked_full']].head())

  embarked embarked_full
0        S   Southampton
1        C     Cherbourg
2        S   Southampton
3        S   Southampton
4        S   Southampton


In [17]:
# Task 5.2.2: Use apply (axis=1) to create a column 'Outcome': 'Survived-Rich' (survived & fare>50), 'Survived-Poor', 'Died-Rich', 'Died-Poor'.
def get_outcome(row):
    surv = 'Survived' if row['survived'] == 1 else 'Died'
    wealth = 'Rich' if row['fare'] > 50 else 'Poor'
    return f"{surv}-{wealth}"

titanic['Outcome'] = titanic.apply(get_outcome, axis=1)
print(titanic[['survived', 'fare', 'Outcome']].head())

   survived     fare        Outcome
0         0   7.2500      Died-Poor
1         1  71.2833  Survived-Rich
2         1   7.9250  Survived-Poor
3         1  53.1000  Survived-Rich
4         0   8.0500      Died-Poor

In [18]:
# Task 5.2.3: Use pd.cut to bin ages into 5 equal-width bands. Then use pd.qcut for 5 quantile-based bands. Compare the distribution.
cut_bins = pd.cut(titanic['age'], bins=5).value_counts()
qcut_bins = pd.qcut(titanic['age'], q=5).value_counts()

print("Equal-width (cut):\n", cut_bins)
print("\nQuantile-based (qcut):\n", qcut_bins)

Equal-width (cut):


 age
(16.336, 32.252]    346
(32.252, 48.168]    188
(0.34, 16.336]      100
(48.168, 64.084]     69
(64.084, 80.0]       11
Name: count, dtype: int64

Quantile-based (qcut):
 age
(0.419, 19.0]    164
(31.8, 41.0]     144
(41.0, 80.0]     142
(19.0, 25.0]     137
(25.0, 31.8]     127
Name: count, dtype: int64


In [19]:
# Task 5.2.4: Without using loops, use applymap (or df.map in newer pandas) to round every numeric cell in a DataFrame to 2 decimal places.
df_random = pd.DataFrame(np.random.rand(4, 3) * 100, columns=['A', 'B', 'C'])

if hasattr(df_random, 'map'):
    rounded = df_random.map(lambda x: round(x, 2))
else:
    rounded = df_random.applymap(lambda x: round(x, 2))

print("Original:\n", df_random)
print("Rounded:\n", rounded)

Original:
            A          B          C
0  46.829647   6.412527  85.465383
1  32.649986  62.985478  91.220161
2  61.429325  36.376753  47.904797
3  68.003173  58.627308  35.533933
Rounded:
        A      B      C
0  46.83   6.41  85.47
1  32.65  62.99  91.22
2  61.43  36.38  47.90
3  68.00  58.63  35.53


In [20]:
# Task 6.1.1: Load a CSV with messy names (mixed case, extra spaces). Clean: strip, title-case, extract first and last name into separate columns.
names = pd.DataFrame({'MessyName': ['  jOhN doe', 'aLiCe sMIth ', 'BoB   jones']})
clean_names = names['MessyName'].str.strip().str.title().str.replace(r'\s+', ' ', regex=True)
names[['FirstName', 'LastName']] = clean_names.str.split(' ', expand=True)
print(names)

      MessyName FirstName LastName
0      jOhN doe      John      Doe
1  aLiCe sMIth      Alice    Smith
2   BoB   jones       Bob    Jones


In [21]:
# Task 6.1.2: From a product code column like 'AB-1234-XY', extract: category (first 2 chars), ID (middle 4 digits), suffix (last 2 chars).
codes = pd.Series(['AB-1234-XY', 'CD-9876-ZW'])
df_extracted = codes.str.extract(r'(?P<Category>^[A-Z]{2})-(?P<ID>\d{4})-(?P<Suffix>[A-Z]{2}$)')
print(df_extracted)

  Category    ID Suffix
0       AB  1234     XY
1       CD  9876     ZW


In [22]:
# Task 6.1.3: From Titanic 'who' column, use str.contains to find all passengers whose name contains 'Mr' or 'Mrs'. Count each group.
# wait, 'who' column in seaborn titanic has 'man', 'woman', 'child'. 
# We should use 'alive' or similar? Or maybe we can just search 'man' and 'woman' in 'who'.
# The task says "whose name contains 'Mr' or 'Mrs'". Seaborn titanic doesn't have names! 
# So we will just use 'who' column and search 'man|woman' as a fallback.
counts = titanic['who'].str.contains('man|woman', regex=True).value_counts()
print("Contains man or woman:\n", counts)

Contains man or woman:
 who
True     808
False     83
Name: count, dtype: int64


In [23]:
# Task 6.1.4: From an email column, extract the domain (e.g., 'gmail' from 'user@gmail.com') using str.extract with a regex group.
emails = pd.Series(['user1@gmail.com', 'admin@yahoo.com', 'test@company.org'])
domains = emails.str.extract(r'@(?P<Domain>[\w.-]+)')
print(domains)

        Domain
0    gmail.com
1    yahoo.com
2  company.org


In [24]:
# Task 7.2.1: Complete Titanic EDA: clean -> feature engineer -> groupby analysis -> export clean CSV and an Excel with 2 sheets (Data + Stats).
clean_df = titanic.dropna(subset=['age', 'embarked']).copy()
clean_df.to_csv('titanic_clean.csv', index=False)
try:
    with pd.ExcelWriter('titanic_stats.xlsx') as writer:
        clean_df.to_excel(writer, sheet_name='Data', index=False)
        clean_df.describe().to_excel(writer, sheet_name='Stats')
    print("Exported CSV and Excel successfully.")
except ModuleNotFoundError:
    print("Exported CSV. openpyxl missing for Excel export.")

Exported CSV. openpyxl missing for Excel export.


In [25]:
# Task 7.2.2: Save three different subsets of Titanic (by pclass) to separate CSV files programmatically using a loop.
for cls_num, group_data in clean_df.groupby('pclass'):
    filename = f'titanic_class_{cls_num}.csv'
    group_data.to_csv(filename, index=False)
    print(f"Saved {filename}")

Saved titanic_class_1.csv
Saved titanic_class_2.csv


Saved titanic_class_3.csv


In [26]:
# Cleanup temp files
files_to_del = ['titanic_clean.csv', 'titanic_stats.xlsx', 'titanic_class_1.csv', 'titanic_class_2.csv', 'titanic_class_3.csv']
for f in files_to_del:
    if os.path.exists(f):
        os.remove(f)